# Evaluation for RAG

In [10]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(k=6)

In [13]:
retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x00000234362598E0>, search_kwargs={})

In [14]:
retriever.invoke("What is agents?")

[Document(id='eef5d061-5fdc-4d00-84f6-820fe9f1f00b', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [15]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:llama-3.3-70b-versatile")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002343AC86AE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002343A2EBF80>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [16]:
from langsmith import traceable

@traceable
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
    Documents:
    {docs_string}
    """
    ai_msg = llm.invoke([
        {"role":"system","content":instructions},
        {"role":"user","content":question}
    ])
    return {"answer":ai_msg.content,"documents":docs}

In [17]:
rag_bot("What is agents")

{'answer': 'In the context of the provided text, an agent refers to a autonomous entity that uses a Large Language Model (LLM) as its core controller. This type of agent is designed to operate independently, making decisions and taking actions based on its own planning and problem-solving capabilities.\n\nIn general, an agent can be defined as a system or entity that:\n\n1. Perceives its environment through sensors or observations\n2. Has goals or objectives to achieve\n3. Can take actions to achieve those goals\n4. Can adapt to changes in its environment\n\nIn the case of LLM-powered autonomous agents, the LLM serves as the agent\'s "brain," enabling it to process information, reason, and make decisions. The agent can use its LLM to:\n\n* Break down complex tasks into smaller subgoals\n* Reflect on its past actions and refine its approach\n* Learn from its mistakes and improve its performance over time\n* Interact with its environment and other agents\n\nAgents can be applied in vario

Building Test Data for Experiments

In [18]:
from langsmith import Client

client = Client()
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['164daa1e-e057-4d81-9970-616c622f376e',
  '0804bece-b95a-4530-8496-d50b3abe9b4d',
  '987fe512-1568-4168-9fc1-427aeabb4e6b'],
 'count': 3,
 'as_of': '2026-08-01T15:31:18.280275366Z'}

# Evaluators

LLM as the Judge

1) Correctness: Response vs Reference Answer

In [19]:
from typing_extensions import Annotated,TypedDict

#correctness output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    correct: Annotated[bool,...,"True if the answer is correct, False Otherwise"]

#correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 
You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.
Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)


In [20]:
def correctness(inputs: dict,outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""
    QUESTION: {inputs['question']}
    GROUND TRUTH ANSWER: {reference_outputs["answer"]}
    STUDENT ANSWER: {outputs["answer"]}"""

    grade = grader_llm.invoke([
        {"role":"system","content": correctness_instructions},
        {"role":"user","content":answers}
    ])

    return grade["correct"]

2) Answer Relevance: Is generated answer relevant to the question

In [21]:
# relevance output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    relevant: Annotated[bool,...,"Provide the score on wether the answer addresses the question"]

# relevance prompt
relevance_instructions="""You are a teacher grading a quiz. 
You will be given a QUESTION and a STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION
Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""


relevance_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

In [22]:
def relevance(inputs: dict,outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness"""
    answer = f"QUESTION: {inputs['question']}\n STUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role":"system","content":relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade['relevant']

3) Groundedness: Response vs Retrieved Docs

does generated answer contain the grounded truth / info from retrieved docs

In [23]:
# groundedness output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    grounded: Annotated[bool,...,"Provide the score on if the answer hallucinates from the documents"]

# Groundedness prompt
grounded_instructions = """You are a teacher grading a quiz. 
You will be given FACTS and a STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.
Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

grounded_llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0).with_structured_output(GroundedGrade,method="json_schema",strict=True)

In [24]:
def groundedness(inputs: dict,outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundness"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role":"system","content":grounded_instructions},
        {"role":"user","content":answer}
    ])
    return grade["grounded"]

4) Retrieval Relevance: Retrieved Docs vs Input

are retrieved docs relevant to the input question

In [25]:
# RetrievalRelevance Output Schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    relevance: Annotated[bool,...,"True if the retrieved documents are relevant to the question, False otherwise"]

# Retrieval Relevance Prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 
You will be given a QUESTION and a set of FACTS provided by the student. 
Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met
Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0).with_structured_output(RetrievalRelevanceGrade,method="json_schema",strict=True)

In [26]:
def retrieval_relevance(inputs: dict,outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\n QUESTION: {inputs['question']}"

    grade = retrieval_relevance_llm.invoke([
        {"role":"system","content":retrieval_relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade["relevant"]

Running the Evaluation

In [27]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness,groundedness,relevance,retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version":"LCEL context, openai/gpt-oss-120b-preview"}
)

experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-25d43cd0' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/64d11d2d-ebfa-46e3-89e6-98628fc83410/compare?selectedSessions=4ffd8f35-f3b0-4328-8310-1aa033f94ce5




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator retrieval_relevance> on run 019fbdf3-86ce-70a0-bcbd-95d04b1c4a83: KeyError('relevant')
Traceback (most recent call last):
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\evaluation\_runner.py", line 1672, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\evaluation\evaluator.py", line 370, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\evaluation\evaluator.py", line 791, in wrapper
    return func(*args, *

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 712, in start
    self.io_loop.

AttributeError: _ARRAY_API not found

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"According to the text, the types of biases tha...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,False,True,None,0.569001,0804bece-b95a-4530-8496-d50b3abe9b4d,019fbdf3-86ce-70a0-bcbd-95d04b1c4a83
1,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by iterat...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",False,False,False,None,2.042340,164daa1e-e057-4d81-9970-616c622f376e,019fbdf3-96c3-7be1-87b8-6ba15ed51cb8
2,What are five types of adversarial attacks?,"According to the text, the five types of adver...",[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,None,0.660911,987fe512-1568-4168-9fc1-427aeabb4e6b,019fbdf3-e376-7943-a902-381d04917f27
